## Working with RAG 

### Basic Workflow of RAG -> 
1. First we need to load the all the folder/ subfolders into the documents object of langchain for this we use the langchain_community.documents_loader import DirectoryLoader, TextLoader 
2. Then we do the chunking RecursiveCharacterTextSplitter and then use the chunk = text_splitter.split_documents(documents)

In [10]:
import os
import glob 
import tiktoken
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI
import tiktoken
from langchain_community.document_loaders import  DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from sklearn.manifold import TSNE
import plotly.graph_objects as go
import glob
from langchain_huggingface import HuggingFaceEmbeddings
from google import genai
import huggingface_hub

In [3]:
model_name = "gemini-2.5-flash"
db_name = "vector_db"

In [4]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path,recursive=True)

print(f"Total file found in knowledge : {len(files)}")

entire_knowledge_base = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"
    
print(f"Total character in entire knowledge base : {len(entire_knowledge_base)}")



Total file found in knowledge : 76
Total character in entire knowledge base : 304434


In [5]:
encoding = tiktoken.encoding_for_model("gpt-4.1-nano")
token = encoding.encode(str(entire_knowledge_base))
print(f"The total number of token in entire knowledge base : {len(token)}")


The total number of token in entire knowledge base : 840170


In [ ]:
client = genai.Client()
response = client.models.count_tokens(
    model = "model_name",
    contents = entire_knowledge_base

)
token_count = response.total_tokens
print(f"Total token for {model_name}: {token_count}")

### Below we have use  langchain_community.document_loaders import  DirectoryLoader, TextLoader

In [6]:
folders = glob.glob("knowledge-base/*")




documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_doc = loader.load()
    for doc in folder_doc:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap =200)
chunk = text_splitter.split_documents(documents)
print(f"The total chunks : {len(chunk)}")
print(type(chunk))



The total chunks : 413
<class 'list'>


In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name,embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunk, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

c:\Users\Sushant\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sushant\Documents\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sushant\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o

Vectorstore created with 413 documents
